# M19a — Core validation-safe-region transfer

**Author:** Ildefons Magrans de Abril  
**Affiliation:** Universitat Politècnica de Catalunya - BarcelonaTech (UPC)

**Purpose.** Reproduce the core question: does a connected temperature region selected only from validation data retain useful operating points on held-out test data?

**Provenance.** The original June 2026 notebook binary is no longer available. The historical manuscript summary is preserved exactly in `results/frozen/historical_reference_metrics.csv`. This publication notebook also runs an **independent protocol replication** using the later fully preserved synthetic task generator. The recovered original settings `TRIALS=12` and seed `20260621` are retained; the exact original task composition was not recoverable and is therefore not claimed to be identical.

In [1]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import tcr_core as tcr

FROZEN = ROOT / "results" / "frozen"
REPRO = ROOT / "results" / "reproduced"
REPRO.mkdir(parents=True, exist_ok=True)

## Configuration

In [2]:
SEED = 20260621
TRIALS = 12
TASKS = ["controlled_d20_white_plus_distractor", "memory_d10", "narma10", "lorenz_x"]
CONFIG = dict(N=60, K=13, lengths=(1200,500,500), washout=100, input_scale=0.8, ridge=1e-5)
print("tasks:", TASKS)
print("trials:", TRIALS, "seed:", SEED)

tasks: ['controlled_d20_white_plus_distractor', 'memory_d10', 'narma10', 'lorenz_x']
trials: 12 seed: 20260621


## Historical manuscript result

In [3]:
hist = pd.read_csv(FROZEN / "historical_reference_metrics.csv")
hist = hist[hist.notebook_id == "M19a"].copy()
display(hist)

,notebook_id,metric,value,ci_low,ci_high,provenance
0,M19a,near_optimal_containment,0.781250,NaN,NaN,historical manuscript result
1,M19a,exact_oracle_containment,0.572917,NaN,NaN,historical manuscript result
2,M19a,safe_gain,0.004702,0.003565,0.005786,historical manuscript result
3,M19a,full_grid_gain,0.030254,NaN,NaN,historical manuscript result


## Independent protocol replication

In [4]:
start=time.time()
rep = tcr.run_panel(TASKS, TRIALS, ["temperature"], seed=SEED, **CONFIG)
rep.to_csv(REPRO / "m19a_replication_case_metrics.csv", index=False)
summary = pd.DataFrame([{
    "n_cases": len(rep),
    "near_optimal_containment": rep.near_contained.mean(),
    "exact_oracle_containment": rep.exact_contained.mean(),
    "mean_safe_gain": rep.safe_gain.mean(),
    "mean_full_grid_gain": rep.full_gain.mean(),
    "mean_safe_width": rep.safe_width.mean(),
}])
summary.to_csv(REPRO / "m19a_replication_summary.csv", index=False)
display(summary.round(6))
print(f"elapsed: {time.time()-start:.2f} s")

,n_cases,near_optimal_containment,exact_oracle_containment,mean_safe_gain,mean_full_grid_gain,mean_safe_width
0,48,0.791667,0.604167,0.001313,0.007873,3.020833


elapsed: 5.81 s


## Interpretation

The historical values are the numbers reported in the manuscript. The independent replication is a fresh audit of the disclosed procedure, not a replacement for the inaccessible original stochastic run. Any numerical difference is therefore reported rather than tuned away.